In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score

# 1. Data Loading and Splitting Function
def load_and_split_data(filename):
    print('Reading file from local path...')
    df = pd.read_csv(filename)

    # Assumes column 0 is ID, columns 1:-1 are features, last column is Target
    X = df.iloc[:, 1:-1].values
    y = df.iloc[:, -1].values
    feature_names = df.columns[1:-1]

    print(f"Original Shape: {df.shape}")
    print(f"X Shape: {X.shape}")
    print(f"y Shape: {y.shape}")

    # --- CHANGE 1: Matched test_size to 0.20 ---
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

    dimensions = X.shape[1]
    return X_train, X_test, y_train, y_test, dimensions, feature_names

def generate_population(pop_size, dimensions):
    population = np.random.randint(2, size=(pop_size, dimensions))
    return population

def selection(population, scores, k=3):
    pop_size = population.shape[0]
    selection_ix = np.random.randint(pop_size)

    for ix in np.random.randint(0, pop_size, k-1):
        if scores[ix] > scores[selection_ix]:
            selection_ix = ix
        elif scores[ix] == scores[selection_ix]:
            if np.count_nonzero(population[ix]) < np.count_nonzero(population[selection_ix]):
                selection_ix = ix
    return selection_ix

def crossover(p1, p2, r_cross):
    pt = int(r_cross * len(p1))
    c1 = np.concatenate((p1[:pt], p2[pt:]))
    c2 = np.concatenate((p2[:pt], p1[pt:]))
    return c1, c2

def mutation(chromosome, r_mut):
    mutated = chromosome.copy()
    for i in range(chromosome.shape[0]):
        if np.random.rand() < r_mut:
            mutated[i] = int(not chromosome[i])
    if np.count_nonzero(mutated) > 0:
        return mutated
    else:
        return chromosome

def evaluate_chromosome(chromosome, classifier, X_train, X_test, y_train, y_test):
    # Ensure chromosome is boolean mask
    bool_mask = chromosome.astype(bool)

    # Feature Selection
    subset_train = X_train[:, bool_mask]
    subset_test = X_test[:, bool_mask]

    classifier.fit(subset_train, y_train)
    ypred = classifier.predict(subset_test)
    score = accuracy_score(y_test, ypred)

    # --- CHANGE 2: Removed print statement to stop spam ---
    # print("Selected features:", np.where(bool_mask)[0])

    return score

def GA(filename, n_iters=10, pop_size=20, r_cross=0.7, r_mut=0.2, random_state=42):

    # Set seed for reproducibility
    np.random.seed(random_state)
    print(f"Random Seed set to: {random_state}")

    X_train, X_test, y_train, y_test, dimensions, feature_names = load_and_split_data(filename)
    population = generate_population(pop_size, dimensions)

    # --- Match Model to Manual Code ---
    classifier = KNeighborsClassifier(n_neighbors=99)

    scores = [evaluate_chromosome(population[i], classifier, X_train, X_test, y_train, y_test) for i in range(pop_size)]
    scores = np.array(scores)

    best_chromosome = population[0]
    best_score = 0

    print(f"Starting Evolution... ({dimensions} total features)")

    for iter_ in range(n_iters):
        selected_indices = [selection(population, scores) for _ in range(pop_size)]
        selected_chroms = population[selected_indices, :]

        crossover_res = []
        for i in range(0, pop_size, 2):
            p1, p2 = selected_chroms[i], selected_chroms[i+1]
            c1, c2 = crossover(p1, p2, r_cross)
            crossover_res.extend([c1, c2])

        crossover_res = np.array(crossover_res)
        mutation_result = [mutation(chrom, r_mut) for chrom in crossover_res]
        population = np.array(mutation_result)

        scores = [evaluate_chromosome(population[i], classifier, X_train, X_test, y_train, y_test) for i in range(pop_size)]
        scores = np.array(scores)

        current_best_ix = np.argmax(scores)
        if scores[current_best_ix] > best_score:
            best_score = scores[current_best_ix]
            best_chromosome = population[current_best_ix].copy()

        print('Gen {}/{}: Best Test Accuracy = {:.4f}, Features Selected = {}'.format(
            iter_+1, n_iters, best_score, np.count_nonzero(best_chromosome)))

    # --- FINAL EXTENDED REPORT ---
    print("\n" + "="*40)
    print("EVOLUTION COMPLETE")
    print("="*40)

    selected_mask = best_chromosome.astype(bool)
    selected_features = feature_names[selected_mask]

    print(f"Total Features Selected: {len(selected_features)}")
    print(f"Selected Features: {list(selected_features)}")

    X_train_final = X_train[:, selected_mask]
    X_test_final = X_test[:, selected_mask]

    final_clf = KNeighborsClassifier(n_neighbors=99)
    final_clf.fit(X_train_final, y_train)

    train_pred = final_clf.predict(X_train_final)
    test_pred = final_clf.predict(X_test_final)

    def print_metrics(y_true, y_pred, set_name):
        print(f"\n--- {set_name} RESULTS ---")
        # --- CHANGE 3: Formatting to match typical reports ---
        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
        rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

        print(f"Accuracy:  {acc:.2f}")
        # print(f"Precision: {prec:.2f}")
        # print(f"Recall:    {rec:.2f}")
        # print(f"F1 Score:  {f1:.2f}")
        print("Confusion Matrix:")
        print(confusion_matrix(y_true, y_pred))

    print_metrics(y_train, train_pred, "TRAINING")
    print_metrics(y_test, test_pred, "TESTING")
    print("="*40)

# --- RUN THE CODE ---
filename = 'brca.csv'
r_cross = 0.7
r_mut = 0.2
pop_size = 20
n_iters = 20
random_state = 42

try:
    GA(filename, n_iters, pop_size, r_cross, r_mut, random_state)
except FileNotFoundError:
    print(f"Error: The file '{filename}' was not found.")
except ValueError as e:
    print(f"\nError: {e}")

Random Seed set to: 42
Reading file from local path...
Original Shape: (569, 32)
X Shape: (569, 30)
y Shape: (569,)
Starting Evolution... (30 total features)
Gen 1/20: Best Test Accuracy = 0.9386, Features Selected = 12
Gen 2/20: Best Test Accuracy = 0.9474, Features Selected = 9
Gen 3/20: Best Test Accuracy = 0.9474, Features Selected = 9
Gen 4/20: Best Test Accuracy = 0.9474, Features Selected = 9
Gen 5/20: Best Test Accuracy = 0.9474, Features Selected = 9
Gen 6/20: Best Test Accuracy = 0.9474, Features Selected = 9
Gen 7/20: Best Test Accuracy = 0.9474, Features Selected = 9
Gen 8/20: Best Test Accuracy = 0.9474, Features Selected = 9
Gen 9/20: Best Test Accuracy = 0.9474, Features Selected = 9
Gen 10/20: Best Test Accuracy = 0.9474, Features Selected = 9
Gen 11/20: Best Test Accuracy = 0.9474, Features Selected = 9
Gen 12/20: Best Test Accuracy = 0.9474, Features Selected = 9
Gen 13/20: Best Test Accuracy = 0.9474, Features Selected = 9
Gen 14/20: Best Test Accuracy = 0.9474, Feat

In [ ]:
import pandas as pd

# Load the dataset to define 'df'
df = pd.read_csv('brca.csv')

# Correcting column names to match the Scikit-learn breast cancer dataset
features = df[[
    'x.texture_mean', 'x.concave_pts_mean', 'x.symmetry_mean', 'x.fractal_dim_mean', 'x.texture_se', 'x.perimeter_se', 'x.area_se', 'x.smoothness_se', 'x.concavity_se', 'x.symmetry_se', 'x.fractal_dim_se', 'x.radius_worst', 'x.perimeter_worst', 'x.smoothness_worst', 'x.compactness_worst', 'x.concave_pts_worst'
]]
label = df['y']  # Note: Based on previous cells, the target column in brca.csv is likely 'y'

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(features, label, test_size= 0.2, random_state= 42)

print('the number of training records',X_train.count())
print('the number of testind records',X_test.count())

the number of training records x.texture_mean         455
x.concave_pts_mean     455
x.symmetry_mean        455
x.fractal_dim_mean     455
x.texture_se           455
x.perimeter_se         455
x.area_se              455
x.smoothness_se        455
x.concavity_se         455
x.symmetry_se          455
x.fractal_dim_se       455
x.radius_worst         455
x.perimeter_worst      455
x.smoothness_worst     455
x.compactness_worst    455
x.concave_pts_worst    455
dtype: int64
the number of testind records x.texture_mean         114
x.concave_pts_mean     114
x.symmetry_mean        114
x.fractal_dim_mean     114
x.texture_se           114
x.perimeter_se         114
x.area_se              114
x.smoothness_se        114
x.concavity_se         114
x.symmetry_se          114
x.fractal_dim_se       114
x.radius_worst         114
x.perimeter_worst      114
x.smoothness_worst     114
x.compactness_worst    114
x.concave_pts_worst    114
dtype: int64


In [ ]:
model = KNeighborsClassifier(n_neighbors= 99)

model.fit(X_train, y_train)
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)
#y_pred = model.predict(X_test)

In [ ]:
# Calculate and display accuracy, precision, recall, and F1 score
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# We specify pos_label='M' because the classes are strings ('B', 'M') instead of (0, 1)
accuracy_train = accuracy_score(y_train, y_train_pred)
precision_train = precision_score(y_train, y_train_pred, pos_label='M')
recall_train= recall_score(y_train, y_train_pred, pos_label='M')
f1_train = f1_score(y_train, y_train_pred, pos_label='M')

print(f'Training Results:')
print(f'Accuracy: {accuracy_train:.4f}')
print(f'Precision: {precision_train:.4f}')
print(f'Recall: {recall_train:.4f}')
print(f'F1 Score: {f1_train:.4f}')

Training Results:
Accuracy: 0.9275
Precision: 0.9722
Recall: 0.8284
F1 Score: 0.8946


In [ ]:
# Calculate and display accuracy, precision, recall, and F1 score for the test set
accuracy_test = accuracy_score(y_test, y_test_pred)
precision_test = precision_score(y_test, y_test_pred, pos_label='M')
recall_test = recall_score(y_test, y_test_pred, pos_label='M')
f1_test = f1_score(y_test, y_test_pred, pos_label='M')

print(f'Testing Results:')
print(f'Accuracy: {accuracy_test:.4f}')
print(f'Precision: {precision_test:.4f}')
print(f'Recall: {recall_test:.4f}')
print(f'F1 Score: {f1_test:.4f}')

Testing Results:
Accuracy: 0.9474
Precision: 0.9512
Recall: 0.9070
F1 Score: 0.9286


In [ ]:
######## Decision Tree  #######

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score

# 1. Data Loading, Splitting, and Scaling Function
def load_split_and_scale_data(filename):
    print('Reading file from local path...')
    df = pd.read_csv(filename)

    # Assumes column 0 is ID, columns 1:-1 are features, last column is Target
    X = df.iloc[:, 1:-1].values
    y = df.iloc[:, -1].values
    feature_names = df.columns[1:-1]

    print(f"Original Shape: {df.shape}")
    print(f"X Shape: {X.shape}")
    print(f"y Shape: {y.shape}")

    # --- MATCHED: test_size=0.20 to match your manual code ---
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

    # # --- ADDED: Standard Scaling ---
    # # We scale here once to avoid re-scaling inside the loop 1000 times.
    # sc = StandardScaler()
    # X_train = sc.fit_transform(X_train)
    # X_test = sc.transform(X_test)
    # print("Data Scaled using StandardScaler.")

    dimensions = X.shape[1]
    return X_train, X_test, y_train, y_test, dimensions, feature_names

def generate_population(pop_size, dimensions):
    population = np.random.randint(2, size=(pop_size, dimensions))
    return population

def selection(population, scores, k=3):
    pop_size = population.shape[0]
    selection_ix = np.random.randint(pop_size)

    for ix in np.random.randint(0, pop_size, k-1):
        if scores[ix] > scores[selection_ix]:
            selection_ix = ix
        elif scores[ix] == scores[selection_ix]:
            if np.count_nonzero(population[ix]) < np.count_nonzero(population[selection_ix]):
                selection_ix = ix
    return selection_ix

def crossover(p1, p2, r_cross):
    pt = int(r_cross * len(p1))
    c1 = np.concatenate((p1[:pt], p2[pt:]))
    c2 = np.concatenate((p2[:pt], p1[pt:]))
    return c1, c2

def mutation(chromosome, r_mut):
    mutated = chromosome.copy()
    for i in range(chromosome.shape[0]):
        if np.random.rand() < r_mut:
            mutated[i] = int(not chromosome[i])
    if np.count_nonzero(mutated) > 0:
        return mutated
    else:
        return chromosome

def evaluate_chromosome(chromosome, classifier, X_train, X_test, y_train, y_test):
    # Ensure chromosome is boolean mask
    bool_mask = chromosome.astype(bool)

    # Feature Selection
    subset_train = X_train[:, bool_mask]
    subset_test = X_test[:, bool_mask]

    classifier.fit(subset_train, y_train)
    ypred = classifier.predict(subset_test)
    score = accuracy_score(y_test, ypred)

    return score

def GA(filename, n_iters=10, pop_size=20, r_cross=0.7, r_mut=0.2, random_state=42):

    # Set seed for reproducibility
    np.random.seed(random_state)
    print(f"Random Seed set to: {random_state}")

    # Load and Scale Data
    X_train, X_test, y_train, y_test, dimensions, feature_names = load_split_and_scale_data(filename)
    population = generate_population(pop_size, dimensions)

    # --- CHANGED HERE: Decision Tree with your specific parameters ---
    classifier = DecisionTreeClassifier(max_depth=3, random_state=42)

    scores = [evaluate_chromosome(population[i], classifier, X_train, X_test, y_train, y_test) for i in range(pop_size)]
    scores = np.array(scores)

    best_chromosome = population[0]
    best_score = 0

    print(f"Starting Evolution... ({dimensions} total features)")

    for iter_ in range(n_iters):
        selected_indices = [selection(population, scores) for _ in range(pop_size)]
        selected_chroms = population[selected_indices, :]

        crossover_res = []
        for i in range(0, pop_size, 2):
            p1, p2 = selected_chroms[i], selected_chroms[i+1]
            c1, c2 = crossover(p1, p2, r_cross)
            crossover_res.extend([c1, c2])

        crossover_res = np.array(crossover_res)
        mutation_result = [mutation(chrom, r_mut) for chrom in crossover_res]
        population = np.array(mutation_result)

        scores = [evaluate_chromosome(population[i], classifier, X_train, X_test, y_train, y_test) for i in range(pop_size)]
        scores = np.array(scores)

        current_best_ix = np.argmax(scores)
        if scores[current_best_ix] > best_score:
            best_score = scores[current_best_ix]
            best_chromosome = population[current_best_ix].copy()

        print('Gen {}/{}: Best Test Accuracy = {:.4f}, Features Selected = {}'.format(
            iter_+1, n_iters, best_score, np.count_nonzero(best_chromosome)))

    # --- FINAL EXTENDED REPORT ---
    print("\n" + "="*40)
    print("EVOLUTION COMPLETE")
    print("="*40)

    selected_mask = best_chromosome.astype(bool)
    selected_features = feature_names[selected_mask]

    print(f"Total Features Selected: {len(selected_features)}")
    print(f"Selected Features: {list(selected_features)}")

    # Subset the SCALED data
    X_train_final = X_train[:, selected_mask]
    X_test_final = X_test[:, selected_mask]

    # Train Final Model on Best Subset
    final_clf = DecisionTreeClassifier(max_depth=3, random_state=42)
    final_clf.fit(X_train_final, y_train)

    train_pred = final_clf.predict(X_train_final)
    test_pred = final_clf.predict(X_test_final)

    def print_metrics(y_true, y_pred, set_name):
        print(f"\n--- {set_name} RESULTS ---")
        acc = accuracy_score(y_true, y_pred)

        print(f"Accuracy:  {acc:.2f}")
        # Note: Precision/Recall/F1 commented out to match your simplified request,
        # but the calculation lines remain if you need them back.
        # prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
        # rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
        # f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

        print("Confusion Matrix:")
        print(confusion_matrix(y_true, y_pred))

    print_metrics(y_train, train_pred, "TRAINING")
    print_metrics(y_test, test_pred, "TESTING")
    print("="*40)

# --- RUN THE CODE ---
filename = 'brca.csv'
r_cross = 0.7
r_mut = 0.2
pop_size = 20
n_iters = 20
random_state = 42

try:
    GA(filename, n_iters, pop_size, r_cross, r_mut, random_state)
except FileNotFoundError:
    print(f"Error: The file '{filename}' was not found.")
except ValueError as e:
    print(f"\nError: {e}")

Random Seed set to: 42
Reading file from local path...
Original Shape: (569, 32)
X Shape: (569, 30)
y Shape: (569,)
Starting Evolution... (30 total features)
Gen 1/20: Best Test Accuracy = 0.9649, Features Selected = 14
Gen 2/20: Best Test Accuracy = 0.9649, Features Selected = 14
Gen 3/20: Best Test Accuracy = 0.9649, Features Selected = 14
Gen 4/20: Best Test Accuracy = 0.9649, Features Selected = 14
Gen 5/20: Best Test Accuracy = 0.9649, Features Selected = 14
Gen 6/20: Best Test Accuracy = 0.9737, Features Selected = 16
Gen 7/20: Best Test Accuracy = 0.9737, Features Selected = 16
Gen 8/20: Best Test Accuracy = 0.9737, Features Selected = 16
Gen 9/20: Best Test Accuracy = 0.9737, Features Selected = 16
Gen 10/20: Best Test Accuracy = 0.9737, Features Selected = 16
Gen 11/20: Best Test Accuracy = 0.9737, Features Selected = 16
Gen 12/20: Best Test Accuracy = 0.9737, Features Selected = 16
Gen 13/20: Best Test Accuracy = 0.9737, Features Selected = 16
Gen 14/20: Best Test Accuracy = 

In [ ]:
import pandas as pd

# Load the dataset to define 'df'
df = pd.read_csv('brca.csv')

# Correcting column names to match the Scikit-learn breast cancer dataset
features = df[[
    'x.radius_mean', 'x.texture_mean', 'x.perimeter_mean', 'x.area_mean', 'x.smoothness_mean', 'x.concave_pts_mean', 'x.symmetry_mean', 'x.radius_se', 'x.area_se', 'x.smoothness_se', 'x.concave_pts_se', 'x.symmetry_se', 'x.radius_worst', 'x.texture_worst', 'x.area_worst', 'x.concavity_worst', 'x.symmetry_worst'
]]
label = df['y']  # Note: Based on previous cells, the target column in brca.csv is likely 'y'

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(features, label, test_size= 0.2, random_state= 42)

print('the number of training records',X_train.count())
print('the number of testind records',X_test.count())

sc = StandardScaler()

X_train_scaled = sc.fit_transform(X_train)
X_test_scaled = sc.transform(X_test)


the number of training records x.radius_mean         455
x.texture_mean        455
x.perimeter_mean      455
x.area_mean           455
x.smoothness_mean     455
x.concave_pts_mean    455
x.symmetry_mean       455
x.radius_se           455
x.area_se             455
x.smoothness_se       455
x.concave_pts_se      455
x.symmetry_se         455
x.radius_worst        455
x.texture_worst       455
x.area_worst          455
x.concavity_worst     455
x.symmetry_worst      455
dtype: int64
the number of testind records x.radius_mean         114
x.texture_mean        114
x.perimeter_mean      114
x.area_mean           114
x.smoothness_mean     114
x.concave_pts_mean    114
x.symmetry_mean       114
x.radius_se           114
x.area_se             114
x.smoothness_se       114
x.concave_pts_se      114
x.symmetry_se         114
x.radius_worst        114
x.texture_worst       114
x.area_worst          114
x.concavity_worst     114
x.symmetry_worst      114
dtype: int64


In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

dt = DecisionTreeClassifier(max_depth= 3, random_state= 42)
dt.fit(X_train_scaled, y_train)

DecisionTreeClassifier(max_depth=3, random_state=42)

In [ ]:
y_train_pred = dt.predict(X_train_scaled)
y_test_pred = dt.predict(X_test_scaled)

In [ ]:
##### Training Data
train_accuracy = accuracy_score(y_train, y_train_pred)
# Specify pos_label='M' since the labels are strings
train_precision = precision_score(y_train, y_train_pred, pos_label='M')
train_recall = recall_score(y_train, y_train_pred, pos_label='M')
train_f1 = f1_score(y_train, y_train_pred, pos_label='M')

##### Testing Data
test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred, pos_label='M')
test_recall = recall_score(y_test, y_test_pred, pos_label='M')
test_f1 = f1_score(y_test, y_test_pred, pos_label='M')

print(f'Training Accuracy: {train_accuracy:.4f}')
print(f'Testing Accuracy:  {test_accuracy:.4f}')

Training Accuracy: 0.9780
Testing Accuracy:  0.9737


In [ ]:
results_df = pd.DataFrame ({
    'Metric':['Accuracy', 'Precision', 'Recal', 'F1_Score'],
    'Training' : [train_accuracy, train_precision, train_recall, train_f1],
    'Testing': [test_accuracy, test_precision, test_recall, test_f1]
})

results_df.T

,0,1,2,3
Metric,Accuracy,Precision,Recal,F1_Score
Training,0.978022,0.970414,0.970414,0.970414
Testing,0.973684,0.97619,0.953488,0.964706


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix

# 1. Data Loading and Splitting Function
def load_and_split_data(filename):
    print('Reading file from local path...')
    df = pd.read_csv(filename)

    # Assumes column 0 is ID, columns 1:-1 are features, last column is Target
    X = df.iloc[:, 1:-1].values
    y = df.iloc[:, -1].values
    feature_names = df.columns[1:-1]

    print(f"Original Shape: {df.shape}")

    # Standard split (20% Test Size)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

    # --- NOTE: SCALING DISABLED AS REQUESTED ---
    # SVMs usually perform better with scaling, but we are keeping raw data
    # to match your specific manual experiment.
    # sc = StandardScaler()
    # X_train = sc.fit_transform(X_train)
    # X_test = sc.transform(X_test)

    dimensions = X.shape[1]
    return X_train, X_test, y_train, y_test, dimensions, feature_names

def generate_population(pop_size, dimensions):
    return np.random.randint(2, size=(pop_size, dimensions))

def selection(population, scores, k=3):
    pop_size = population.shape[0]
    selection_ix = np.random.randint(pop_size)
    for ix in np.random.randint(0, pop_size, k-1):
        if scores[ix] > scores[selection_ix]:
            selection_ix = ix
    return selection_ix

def crossover(p1, p2, r_cross):
    pt = int(r_cross * len(p1))
    return np.concatenate((p1[:pt], p2[pt:])), np.concatenate((p2[:pt], p1[pt:]))

def mutation(chromosome, r_mut):
    mutated = chromosome.copy()
    for i in range(chromosome.shape[0]):
        if np.random.rand() < r_mut:
            mutated[i] = int(not chromosome[i])
    return mutated if np.count_nonzero(mutated) > 0 else chromosome

def evaluate_chromosome(chromosome, classifier, X_train, X_test, y_train, y_test):
    bool_mask = chromosome.astype(bool)
    if not any(bool_mask): return 0.0 # Safety for empty selection

    subset_train = X_train[:, bool_mask]
    subset_test = X_test[:, bool_mask]

    classifier.fit(subset_train, y_train)
    ypred = classifier.predict(subset_test)
    return accuracy_score(y_test, ypred)

# --- CLEANED UP FUNCTION DEFINITION ---
# Removed default values (e.g. n_iters=10) so you only set them at the bottom.
def GA(filename, n_iters, pop_size, r_cross, r_mut, random_state):

    np.random.seed(random_state)
    print(f"Random Seed set to: {random_state}")

    X_train, X_test, y_train, y_test, dimensions, feature_names = load_and_split_data(filename)
    population = generate_population(pop_size, dimensions)

    # --- CHANGED HERE: SVM with Linear Kernel ---
    print("\nInitializing SVM with kernel='linear'...")
    classifier = SVC(kernel='linear', gamma='scale', C=1.0)

    # Initial Evaluation
    scores = np.array([evaluate_chromosome(p, classifier, X_train, X_test, y_train, y_test) for p in population])

    best_chromosome = population[np.argmax(scores)]
    best_score = np.max(scores)

    print(f"Starting Evolution... ({dimensions} total features)")

    for iter_ in range(n_iters):
        # Selection
        selected_indices = [selection(population, scores) for _ in range(pop_size)]
        selected_chroms = population[selected_indices]

        # Crossover
        crossover_res = []
        for i in range(0, pop_size, 2):
            p1, p2 = selected_chroms[i], selected_chroms[i+1]
            c1, c2 = crossover(p1, p2, r_cross)
            crossover_res.extend([c1, c2])

        # Mutation
        population = np.array([mutation(c, r_mut) for c in crossover_res])

        # Evaluation
        scores = np.array([evaluate_chromosome(p, classifier, X_train, X_test, y_train, y_test) for p in population])

        # Track Best
        current_best_ix = np.argmax(scores)
        if scores[current_best_ix] > best_score:
            best_score = scores[current_best_ix]
            best_chromosome = population[current_best_ix].copy()

        print(f'Gen {iter_+1}/{n_iters}: Best Accuracy = {best_score:.4f}, Features = {np.count_nonzero(best_chromosome)}')

    # --- FINAL REPORT ---
    print("\n" + "="*40 + "\nEVOLUTION COMPLETE\n" + "="*40)

    selected_mask = best_chromosome.astype(bool)
    selected_features = feature_names[selected_mask]
    print(f"Total Features Selected: {len(selected_features)}")
    print(f"Selected Features: {list(selected_features)}")

    # Final Training with Best Subset
    X_train_final = X_train[:, selected_mask]
    X_test_final = X_test[:, selected_mask]

    final_clf = SVC(kernel='linear', gamma='scale', C=1.0)
    final_clf.fit(X_train_final, y_train)

    print("\n--- TRAINING RESULTS ---")
    print(f"Accuracy: {accuracy_score(y_train, final_clf.predict(X_train_final)):.2f}")

    print("\n--- TESTING RESULTS ---")
    print(f"Accuracy: {accuracy_score(y_test, final_clf.predict(X_test_final)):.2f}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, final_clf.predict(X_test_final)))
    print("="*40)

# --- CONTROL PANEL (Set your parameters here ONLY) ---
filename = 'brca.csv'

# THESE are the values that actually matter:
PARAMETERS = {
    "n_iters": 20,
    "pop_size": 20,
    "r_cross": 0.8,
    "r_mut": 0.2,
    "random_state": 42
}

# Run the GA
try:
    # We unpack the dictionary (**PARAMETERS) to pass them into the function
    GA(filename, **PARAMETERS)
except FileNotFoundError:
    print(f"Error: The file '{filename}' was not found.")
except Exception as e:
    print(f"\nError: {e}")

Random Seed set to: 42
Reading file from local path...
Original Shape: (569, 32)

Initializing SVM with kernel='linear'...
Starting Evolution... (30 total features)
Gen 1/20: Best Accuracy = 0.9649, Features = 17
Gen 2/20: Best Accuracy = 0.9649, Features = 17
Gen 3/20: Best Accuracy = 0.9649, Features = 17
Gen 4/20: Best Accuracy = 0.9649, Features = 17
Gen 5/20: Best Accuracy = 0.9649, Features = 17
Gen 6/20: Best Accuracy = 0.9649, Features = 17
Gen 7/20: Best Accuracy = 0.9649, Features = 17
Gen 8/20: Best Accuracy = 0.9649, Features = 17
Gen 9/20: Best Accuracy = 0.9649, Features = 17
Gen 10/20: Best Accuracy = 0.9649, Features = 17
Gen 11/20: Best Accuracy = 0.9649, Features = 17
Gen 12/20: Best Accuracy = 0.9649, Features = 17
Gen 13/20: Best Accuracy = 0.9649, Features = 17
Gen 14/20: Best Accuracy = 0.9649, Features = 17
Gen 15/20: Best Accuracy = 0.9649, Features = 17
Gen 16/20: Best Accuracy = 0.9737, Features = 17
Gen 17/20: Best Accuracy = 0.9737, Features = 17
Gen 18/20: 

In [ ]:
import pandas as pd

# Load the dataset to define 'df'
df = pd.read_csv('brca.csv')

# Correcting column names to match the Scikit-learn breast cancer dataset
features = df[[
    'x.radius_mean', 'x.texture_mean', 'x.perimeter_mean', 'x.concave_pts_mean', 'x.symmetry_mean', 'x.fractal_dim_mean', 'x.radius_se', 'x.area_se', 'x.compactness_se', 'x.concavity_se', 'x.concave_pts_se', 'x.symmetry_se', 'x.fractal_dim_se', 'x.radius_worst', 'x.perimeter_worst', 'x.smoothness_worst', 'x.fractal_dim_worst'
]]
label = df['y']  # Note: Based on previous cells, the target column in brca.csv is likely 'y'

In [ ]:
import numpy as np
import pandas as pd
from sklearn import datasets
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(features, label, test_size= 0.2, random_state= 42)

print('the number of training records',X_train.count())
print('the number of testind records',X_test.count())

# Define kernels to test
kernels = ['linear']

for kernel in kernels:
    print(f"\nSVM with {kernel} kernel")

    # Create SVM model with the specified kernel
    model = SVC(kernel=kernel, gamma='scale', C=1.0)

    # Train the model
    model.fit(X_train, y_train)

    # Predict
    y_train_pred= model.predict(X_train)
    y_test_pred = model.predict(X_test)

    ##### Training Data
    train_accuracy = accuracy_score(y_train, y_train_pred)
    # Specify pos_label='M' for string labels
    train_precision = precision_score(y_train, y_train_pred, pos_label='M')
    train_recall = recall_score(y_train, y_train_pred, pos_label='M')
    train_f1 = f1_score(y_train, y_train_pred, pos_label='M')

    ##### Testing Data
    test_accuracy = accuracy_score(y_test, y_test_pred)
    test_precision = precision_score(y_test, y_test_pred, pos_label='M')
    test_recall = recall_score(y_test, y_test_pred, pos_label='M')
    test_f1 = f1_score(y_test, y_test_pred, pos_label='M')

    # Evaluate
    print("Training Accuracy:", train_accuracy)
    print("Testing Accuracy:", test_accuracy)
    print("Confusion Matrix (Train):\n", confusion_matrix(y_train, y_train_pred))
    print("Confusion Matrix (Test):\n", confusion_matrix(y_test, y_test_pred))
    print("Classification Report (Test):\n", classification_report(y_test, y_test_pred))

the number of training records x.radius_mean          455
x.texture_mean         455
x.perimeter_mean       455
x.concave_pts_mean     455
x.symmetry_mean        455
x.fractal_dim_mean     455
x.radius_se            455
x.area_se              455
x.compactness_se       455
x.concavity_se         455
x.concave_pts_se       455
x.symmetry_se          455
x.fractal_dim_se       455
x.radius_worst         455
x.perimeter_worst      455
x.smoothness_worst     455
x.fractal_dim_worst    455
dtype: int64
the number of testind records x.radius_mean          114
x.texture_mean         114
x.perimeter_mean       114
x.concave_pts_mean     114
x.symmetry_mean        114
x.fractal_dim_mean     114
x.radius_se            114
x.area_se              114
x.compactness_se       114
x.concavity_se         114
x.concave_pts_se       114
x.symmetry_se          114
x.fractal_dim_se       114
x.radius_worst         114
x.perimeter_worst      114
x.smoothness_worst     114
x.fractal_dim_worst    114
dtype: i